# PathSnap — Notebook 08: M8 review of the LightGBM pipeline vs the deployed pipeline
**Owner: C (nisarg-ravi).** Before exporting the final model (M8) we compared C's independent LightGBM pipeline
(`alt_lightgbm_pipeline/`, built on stand-in labels/features) with the deployed pipeline (approved M3 labels + 16 features,
hist_gb_tuned_ll + HMM). Everything is measured on the **80 training trips, out-of-fold, against the approved M3 labels**.
Student-test is not used for this decision.

1. Score C's LightGBM out-of-fold probabilities against OUR labels on the same points.
2. Ablation with the SAME model (hist_gb, trial-3 params) and folds: our 16 features / ours + C's 5 extra / C's 15 rebuilt.
3. Does the classifier gain survive HMM decoding (the system-level objective)?
Runtime on Colab ≈ 30 min (rebuild dataset ~2 min, three 5-fold CVs ~10 min, HMM decodes ~15 min).

In [ ]:
import os, sys, json, time, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/PathSnap"
    ZIP_PATH, EXTRACT_DIR = f"{DRIVE}/Student folder.zip", "/content/pathsnap_data"
    M6, OUT = f"{DRIVE}/m6_outputs", f"{DRIVE}/m8_outputs"
    REPO = "/content/01_pathsnap"
    if not os.path.isdir(REPO): os.system(f"git clone -q https://github.com/paya5am/01_pathsnap {REPO}")
    PKG_PARENT = REPO
else:
    ZIP_PATH = os.environ.get("PATHSNAP_ZIP", "Student folder.zip"); EXTRACT_DIR = os.environ.get("PATHSNAP_EXTRACT", "pathsnap_data")
    M6 = os.environ.get("PATHSNAP_M6", "m6_outputs"); OUT = os.environ.get("PATHSNAP_M8", "m8_outputs")
    PKG_PARENT = os.environ.get("PATHSNAP_PKG", ".")
sys.path.insert(0, PKG_PARENT)
C_NPZ = os.path.join(PKG_PARENT, "alt_lightgbm_pipeline", "experiments", "results", "oof_probs_train.npz")
if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(EXTRACT_DIR)
DATA_ROOT = next(dp for dp, dn, _ in os.walk(EXTRACT_DIR) if "train" in dn and "student_test" in dn)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss
from pathsnap.pipeline import process_trip
from pathsnap.io import list_trips, trip_dir
from pathsnap.features import FEATURES, C_EXTRA
from pathsnap.modeling import point_predictions
from pathsnap.hmm import TripLattice, viterbi
os.makedirs(OUT, exist_ok=True)
try: display
except NameError: display = print

In [ ]:
# ---- training dataset with ALL columns (our 16 + C's 5 extra) --------------------------------------
t0 = time.time(); Fs, Ps = [], []
for t in list_trips(DATA_ROOT, "train"):
    f, p, _ = process_trip(trip_dir(DATA_ROOT, "train", t)); Fs.append(f); Ps.append(p)
TR = pd.concat(Fs, ignore_index=True).sort_values(["trip_id", "pt", "dist_rank"], kind="mergesort").reset_index(drop=True)
TR.insert(1, "partition", "train"); PT = pd.concat(Ps, ignore_index=True); PT.insert(1, "partition", "train")
LAB = PT[PT.label_status == 0][["trip_id", "pt", "ref_arc"]]
print(f"{len(TR):,} candidate rows, {len(LAB):,} labelled points, {time.time()-t0:.0f}s")

## 1. C's LightGBM vs our hist_gb, classifier only, scored against OUR labels on the same points

In [ ]:
z = np.load(C_NPZ)
C = pd.DataFrame({"trip_id": z["trip"], "pt": z["point"], "arc": z["arc"], "p_tuned": z["tuned"], "p_untuned": z["untuned"], "y_C": z["y"]})
top = lambda df, col: df.loc[df.groupby(["trip_id", "pt"])[col].idxmax(), ["trip_id", "pt", "arc"]].rename(columns={"arc": "pred"})
OURS = pd.read_csv(f"{M6}/hist_gb_tuned_ll_oof.csv.gz")
agree = LAB.merge(C[C.y_C == 1][["trip_id", "pt", "arc"]].rename(columns={"arc": "ref_C"}), on=["trip_id", "pt"])
S = LAB.merge(top(C, "p_tuned"), on=["trip_id", "pt"]).merge(top(C, "p_untuned"), on=["trip_id", "pt"], suffixes=("_Ct", "_Cu")) \
       .merge(top(OURS, "oof_proba").rename(columns={"pred": "pred_ours"}), on=["trip_id", "pt"])
T1 = pd.DataFrame({"classifier": ["C LightGBM untuned", "C LightGBM tuned", "our hist_gb_tuned_ll"],
                   "top1_under_our_labels": [np.mean(S[c] == S.ref_arc) for c in ["pred_Cu", "pred_Ct", "pred_ours"]]})
pt = S.assign(c=S.pred_Ct == S.ref_arc, o=S.pred_ours == S.ref_arc).groupby("trip_id")[["c", "o"]].mean()
print(f"C stand-in labels agree with approved labels on {np.mean(agree.ref_C == agree.ref_arc):.4f} of shared points; "
      f"evaluation set {len(S):,} points ({len(S)/len(LAB):.3f} of ours)")
display(T1.round(4)); T1.to_csv(f"{OUT}/01_classifier_under_our_labels.csv", index=False)
print(f"per trip: C better in {(pt.c > pt.o).sum()}/80, Wilcoxon p = {wilcoxon(pt.c, pt.o).pvalue:.2e}")

## 2. Ablation: same model (hist_gb trial 3), same folds — which features explain the gap?

In [ ]:
bp = json.load(open(f"{M6}/best_params.json"))["best_logloss_tied_params"]
FIX = dict(max_iter=600, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, max_bins=255, random_state=42)
C_SET = ["dist", "dist_rel", "dist_rank", "n_cands", "d_nearest", "head_diff_raw", "cos_heading", "speed", "heading_known",
         "arc_len_log", "two_way", "proj_t", "dist_prev3", "dist_next3", "head_diff_x_moving2"]
SETS = {"ours_16 (deployed)": FEATURES, "ours_16 + C_extra_5": FEATURES + C_EXTRA, "C_15 rebuilt in our pipeline": C_SET}
FOLDS = list(GroupKFold(5).split(TR, groups=TR.cv_group.values))
nlab = PT[PT.label_status == 0].groupby("trip_id").size(); OOFS = {}; rows = []
for name, cols in SETS.items():
    oof = np.zeros(len(TR))
    for tr_i, va_i in FOLDS:
        d = TR.iloc[tr_i]; d = d[d.label >= 0]
        oof[va_i] = HistGradientBoostingClassifier(**bp, **FIX).fit(d[cols].values, d.label.values).predict_proba(TR.iloc[va_i][cols].values)[:, 1]
    pp = point_predictions(TR, oof); pp = pp[pp.label_status == 0]; l = TR.label.values >= 0
    rows.append({"feature_set": name, "n_features": len(cols), "oof_top1": pp.pred_is_ref.sum() / nlab.sum(),
                 "oof_log_loss": log_loss(TR.label.values[l], np.clip(oof[l], 1e-7, 1 - 1e-7))})
    OOFS[name] = pd.DataFrame({"trip_id": TR.trip_id, "pt": TR.pt, "arc": TR.arc, "p": oof}); print(rows[-1], flush=True)
T2 = pd.DataFrame(rows); T2.to_csv(f"{OUT}/02_feature_ablation.csv", index=False); display(T2.round(4))

## 3. Does the gain survive the HMM? (same intersection of points for everyone; β tuned per source on train OOF)

In [ ]:
inter = set(map(tuple, C[["trip_id", "pt"]].drop_duplicates().values))
SRC = {"ours_16 (deployed)": (OOFS["ours_16 (deployed)"], [3]), "ours_16 + C_extra_5": (OOFS["ours_16 + C_extra_5"], [3]),
       "C LightGBM tuned": (C.rename(columns={"p_tuned": "p"}), [2, 3, 5, 10, 20, 40])}
per, rows = {}, []
for name, (O, betas) in SRC.items():
    acc = {b: [] for b in betas}
    for t in list_trips(DATA_ROOT, "train"):          # one lattice at a time (memory-safe)
        o = O[O.trip_id == t]
        lat = TripLattice(trip_dir(DATA_ROOT, "train", t), dict(zip(zip(o.pt.values, o.arc.values), o.p.values)))
        q = PT[PT.trip_id == t].sort_values("pt"); m = (q.label_status.values == 0) & np.array([(t, x) in inter for x in q.pt.values])
        for b in betas:
            pr, _ = viterbi(lat, beta=b, break_s=300, gap_exp=0)
            acc[b].append((t, int((pr[m] == q.ref_arc.values[m]).sum()), int(m.sum())))
        del lat
    scores = {b: sum(x[1] for x in v) / sum(x[2] for x in v) for b, v in acc.items()}
    b = max(scores, key=scores.get)
    R = pd.DataFrame(acc[b], columns=["trip", "ok", "n"]).set_index("trip")
    per[name] = R.ok / R.n; rows.append({"emissions": name, "hmm_top1_pooled": scores[b], "beta": b})
    print(name, {k: round(v, 4) for k, v in scores.items()}, flush=True)
T3 = pd.DataFrame(rows)
for k in ["ours_16 + C_extra_5", "C LightGBM tuned"]:
    d = per[k] - per["ours_16 (deployed)"]
    T3.loc[T3.emissions == k, ["per_trip_mean_diff", "trips_better", "trips_worse", "wilcoxon_p"]] = [d.mean(), (d > 0).sum(), (d < 0).sum(), wilcoxon(per[k], per["ours_16 (deployed)"]).pvalue]
T3.to_csv(f"{OUT}/03_after_hmm.csv", index=False); display(T3.round(4))
print("Decision (D31/D32): no alternative beats the deployed pipeline consistently after HMM (pooled and per-trip disagree, "
      "LightGBM p≈0.77) → export hist_gb_tuned_ll on the approved 16 features. C's extra features add +3.1 points at classifier "
      "level, but they carry temporal context that the HMM already provides.")